# Fingerprint benchmark

Source: archived main notebook, cells 62–70. This notebook retains every available representation and fits PCA within each training fold. It reports a fixed-parameter baseline and a separate nested TPE search for the representation used in the archived benchmark.


In [ ]:
from pathlib import Path
import os
import sys

def repository_root(start):
    for path in (start, *start.parents):
        if (path / "Code" / "model_train.py").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the CycloAddGenerator repository.")

REPO_ROOT = repository_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks" / "03_modeling"))
DATA_DIR = REPO_ROOT / "Data" / "DFT_Result"
DESCRIPTOR_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_.pkl"

In [ ]:
import pickle
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK, Trials, fmin, hp, space_eval, tpe
from sklearn.decomposition import PCA
from sklearn.ensemble import (AdaBoostRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, RandomForestRegressor)
from sklearn.linear_model import BayesianRidge, Lasso, Ridge, SGDRegressor
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor
from _modeling_common import metrics, new_catboost

bundle_path = DATA_DIR / "Descriptors" / "ZINC_fingerprint_bundle.pkl"
with bundle_path.open("rb") as handle:
    bundle = pickle.load(handle)
fingerprints, targets = bundle["features"], bundle["targets"]
row_count = len(bundle["reaction_keys"])
if any(len(x) != row_count for x in fingerprints.values()):
    raise ValueError("The fingerprint matrices contain different reactions")


In [ ]:
MODEL_FACTORIES = {
    "GB": lambda: GradientBoostingRegressor(random_state=0),
    "RF": lambda: RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1),
    "ADB": lambda: AdaBoostRegressor(n_estimators=200, random_state=0),
    "ET": lambda: ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1),
    "XGB": lambda: XGBRegressor(n_estimators=200, max_depth=5, random_state=0),
    "Lasso": lambda: Lasso(alpha=0.5), "Ridge": lambda: Ridge(alpha=0.5),
    "BR": lambda: BayesianRidge(), "SGD": lambda: SGDRegressor(random_state=0),
    "SVR": lambda: SVR(C=1.0), "KNN": lambda: KNeighborsRegressor(n_neighbors=5),
    "CatBoost": new_catboost,
}

def estimator_for(name, n_rows, n_features, params=None):
    steps = []
    if n_features > 300:
        steps.append(PCA(n_components=min(300, n_rows - 1, n_features),
                         whiten=True, random_state=0))
    if name in {"Lasso", "Ridge", "BR", "SGD", "SVR", "KNN"}:
        steps.append(StandardScaler())
    estimator = MODEL_FACTORIES[name]()
    if params:
        estimator.set_params(**params)
    steps.append(estimator)
    return make_pipeline(*steps)


## Fixed-parameter five-fold baseline

This fast comparison uses preset estimator parameters. Select fewer models or representations before running if only a subset is needed. Hyperopt results are reported separately below.


In [ ]:
SELECTED_MODELS = list(MODEL_FACTORIES)
SELECTED_REPRESENTATIONS = list(fingerprints)
folds = list(KFold(5, shuffle=True, random_state=0).split(np.arange(row_count)))
rows = []
for representation in SELECTED_REPRESENTATIONS:
    x = np.asarray(fingerprints[representation], dtype=float)
    for target, y in targets.items():
        y = np.asarray(y, dtype=float)
        for name in SELECTED_MODELS:
            for fold, (train_ids, test_ids) in enumerate(folds):
                model = estimator_for(name, len(train_ids), x.shape[1])
                model.fit(x[train_ids], y[train_ids])
                rows.append({"representation": representation, "target": target,
                             "model": name, "fold": fold,
                             **metrics(y[test_ids], model.predict(x[test_ids]))})
benchmark = pd.DataFrame(rows)
display(benchmark.groupby(["target", "representation", "model"])[["r2", "mae"]].mean())


## Nested Hyperopt tuning

The archived benchmark used TPE with 20 evaluations per model and target. Here, each outer test fold is held out from parameter selection: TPE maximizes mean inner-fold R² using only the outer training data, then the selected estimator is refit on that training data and evaluated on the untouched outer test fold. The default representation, `PhysOrg + SVO`, matches the `X = X_Ga` representation selected in the archived notebook. Add other keys from `fingerprints` to `HYPEROPT_REPRESENTATIONS` to tune them as well. CatBoost remains a fixed-parameter comparator because it was not included in the archived search. With the defaults, one representation requires about 22,000 inner-fold model fits; reduce the model, target, or evaluation lists for a smaller run.


In [ ]:
N_ESTIMATORS = [100, 200, 300]
MIN_SAMPLES_SPLIT = [2, 3, 4, 5, 6]
MIN_SAMPLES_LEAF = [1, 2, 3, 4]
MAX_DEPTH = [2, 3, 4, 5, 6]
RIDGE_ALPHA = [0.0, 0.5, 1.0]
BR_PRIOR = [1e-7, 1e-6, 1e-5]

HYPEROPT_SPACES = {
    "GB": {
        "n_estimators": hp.choice("GB_n_estimators", N_ESTIMATORS),
        "min_samples_split": hp.choice("GB_min_samples_split", MIN_SAMPLES_SPLIT),
        "min_samples_leaf": hp.choice("GB_min_samples_leaf", MIN_SAMPLES_LEAF),
    },
    "RF": {
        "n_estimators": hp.choice("RF_n_estimators", N_ESTIMATORS),
        "min_samples_split": hp.choice("RF_min_samples_split", MIN_SAMPLES_SPLIT),
        "min_samples_leaf": hp.choice("RF_min_samples_leaf", MIN_SAMPLES_LEAF),
    },
    "ADB": {"n_estimators": hp.choice("ADB_n_estimators", N_ESTIMATORS)},
    "ET": {
        "n_estimators": hp.choice("ET_n_estimators", N_ESTIMATORS),
        "min_samples_split": hp.choice("ET_min_samples_split", MIN_SAMPLES_SPLIT),
        "min_samples_leaf": hp.choice("ET_min_samples_leaf", MIN_SAMPLES_LEAF),
    },
    "XGB": {
        "n_estimators": hp.choice("XGB_n_estimators", N_ESTIMATORS),
        "max_depth": hp.choice("XGB_max_depth", MAX_DEPTH),
    },
    "Lasso": {"alpha": hp.choice("Lasso_alpha", RIDGE_ALPHA)},
    "Ridge": {"alpha": hp.choice("Ridge_alpha", RIDGE_ALPHA)},
    "BR": {
        "alpha_1": hp.choice("BR_alpha_1", BR_PRIOR),
        "alpha_2": hp.choice("BR_alpha_2", BR_PRIOR),
        "lambda_1": hp.choice("BR_lambda_1", BR_PRIOR),
        "lambda_2": hp.choice("BR_lambda_2", BR_PRIOR),
    },
    "SGD": {"alpha": hp.choice("SGD_alpha", RIDGE_ALPHA)},
    "SVR": {
        # C=0 from the archived space is invalid for scikit-learn's SVR.
        "C": hp.choice("SVR_C", [0.5, 1.0]),
        "epsilon": hp.choice("SVR_epsilon", [0.1, 0.2, 0.3]),
    },
    "KNN": {"n_neighbors": hp.choice("KNN_n_neighbors", list(range(1, 11)))},
}

HYPEROPT_REPRESENTATIONS = ["PhysOrg + SVO"]
HYPEROPT_MODELS = list(HYPEROPT_SPACES)
HYPEROPT_TARGETS = list(targets)
TPE_MAX_EVALS = 20
INNER_SPLITS = 5
OUTER_SPLITS = 5
TPE_RANDOM_STATE = 0

def tune_on_training_fold(name, x_train, y_train, random_state):
    """Select parameters using only inner folds of one outer training fold."""
    space = HYPEROPT_SPACES[name]
    inner_folds = list(KFold(
        n_splits=INNER_SPLITS, shuffle=True, random_state=random_state
    ).split(np.arange(len(y_train))))

    def objective(params):
        scores = []
        for inner_train, inner_valid in inner_folds:
            model = estimator_for(name, len(inner_train), x_train.shape[1], params)
            model.fit(x_train[inner_train], y_train[inner_train])
            prediction = model.predict(x_train[inner_valid])
            scores.append(metrics(y_train[inner_valid], prediction)["r2"])
        return {"loss": -float(np.mean(scores)), "status": STATUS_OK}

    best_indices = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=TPE_MAX_EVALS,
        trials=Trials(),
        rstate=np.random.default_rng(random_state),
        show_progressbar=False,
    )
    return space_eval(space, best_indices)


In [ ]:
if not set(HYPEROPT_REPRESENTATIONS).issubset(fingerprints):
    raise KeyError("A requested Hyperopt representation is absent from the bundle")
if not set(HYPEROPT_MODELS).issubset(MODEL_FACTORIES):
    raise KeyError("A requested Hyperopt model has no estimator factory")
if not set(HYPEROPT_TARGETS).issubset(targets):
    raise KeyError("A requested Hyperopt target is absent from the bundle")

outer_folds = list(KFold(
    n_splits=OUTER_SPLITS, shuffle=True, random_state=0
).split(np.arange(row_count)))
tuned_rows = []
for representation in HYPEROPT_REPRESENTATIONS:
    x = np.asarray(fingerprints[representation], dtype=float)
    for target in HYPEROPT_TARGETS:
        y = np.asarray(targets[target], dtype=float)
        for name in HYPEROPT_MODELS:
            for outer_fold, (train_ids, test_ids) in enumerate(outer_folds):
                search_seed = TPE_RANDOM_STATE + outer_fold
                best_params = tune_on_training_fold(
                    name, x[train_ids], y[train_ids], search_seed
                )
                model = estimator_for(
                    name, len(train_ids), x.shape[1], best_params
                )
                model.fit(x[train_ids], y[train_ids])
                prediction = model.predict(x[test_ids])
                tuned_rows.append({
                    "representation": representation,
                    "target": target,
                    "model": name,
                    "outer_fold": outer_fold,
                    "best_params": best_params,
                    **metrics(y[test_ids], prediction),
                })

nested_hyperopt = pd.DataFrame(tuned_rows)
hyperopt_summary = nested_hyperopt.groupby(
    ["target", "representation", "model"]
)[["r2", "mae"]].agg(["mean", "std"])
display(hyperopt_summary)
display(nested_hyperopt.head())
